In [1]:
import os
os.listdir()

['.ai-navigator',
 '.anaconda',
 '.bash_history',
 '.cache',
 '.conda',
 '.condarc',
 '.continuum',
 '.git',
 '.gitconfig',
 '.idlerc',
 '.ipynb_checkpoints',
 '.ipython',
 '.jupyter',
 '.mamba',
 '.matplotlib',
 '.ms-ad',
 '.packettracer',
 '.redhat',
 '.streamlit',
 '.UGENE_files',
 '.viminfo',
 '.VirtualBox',
 '.virtual_documents',
 '.vscode',
 '00_convert_rds',
 '31_xgboost_feature_importance_plot.R',
 '33_project_abstract.txt',
 '34_download_cnv.R',
 '35_prepare_cnv_matrix.R',
 '36_merge_multiomics_cnv.R',
 '37_build_multiomics_cnv_model.R',
 '38_cnv_kaplan_meier.R',
 '39_compare_cnv_vs_no_cnv.R',
 '40_train_test_validation__OLD.R',
 '41_train_test_kaplan_meier__OLD.R',
 '42_train_test_report.R',
 '43_repeated_train_test_cindex.R',
 '44_time_dependent_ROC.R',
 '45_final_model_summary.R',
 'anaconda3',
 'anaconda4',
 'anaconda4a',
 'anaconda5',
 'anaconda5a',
 'anaconda_projects',
 'AppData',
 'Application Data',
 'bidmc_01_Numerics.csv',
 'bidmc_01_Signals.csv',
 'bidmc_02_Breaths

In [2]:
import pandas as pd
df = pd.read_csv("bidmc_01_Numerics.csv")
df.head()

,Time [s],HR,PULSE,RESP,SpO2
0,0,94,93.0,25,97.0
1,1,94,93.0,25,97.0
2,2,94,93.0,25,97.0
3,3,92,93.0,26,97.0
4,4,93,93.0,26,97.0


In [3]:
df.head()




,Time [s],HR,PULSE,RESP,SpO2
0,0,94,93.0,25,97.0
1,1,94,93.0,25,97.0
2,2,94,93.0,25,97.0
3,3,92,93.0,26,97.0
4,4,93,93.0,26,97.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Time [s]  481 non-null    int64  
 1    HR       481 non-null    int64  
 2    PULSE    468 non-null    float64
 3    RESP     481 non-null    int64  
 4    SpO2     468 non-null    float64
dtypes: float64(2), int64(3)
memory usage: 18.9 KB


In [5]:
df.describe()


,Time [s],HR,PULSE,RESP,SpO2
count,481.000000,481.000000,468.000000,481.000000,468.000000
mean,240.000000,91.318087,91.008547,21.438669,96.916667
std,138.997002,1.012765,1.137229,1.296181,0.357693
min,0.000000,88.000000,89.000000,20.000000,96.000000
25%,120.000000,91.000000,90.000000,21.000000,97.000000
50%,240.000000,91.000000,91.000000,21.000000,97.000000
75%,360.000000,92.000000,91.000000,22.000000,97.000000
max,480.000000,95.000000,94.000000,26.000000,98.000000


In [6]:
print(df.columns)

Index(['Time [s]', ' HR', ' PULSE', ' RESP', ' SpO2'], dtype='object')


In [7]:
import pandas as pd

df = pd.read_csv("bidmc_01_Numerics.csv")

print("PRZED:", df.columns)

df.columns = df.columns.str.strip()

print("PO:", df.columns)

df[["SpO2", "HR", "RESP"]].head()

PRZED: Index(['Time [s]', ' HR', ' PULSE', ' RESP', ' SpO2'], dtype='object')
PO: Index(['Time [s]', 'HR', 'PULSE', 'RESP', 'SpO2'], dtype='object')


,SpO2,HR,RESP
0,97.0,94,25
1,97.0,94,25
2,97.0,94,25
3,97.0,92,26
4,97.0,93,26


In [8]:
df["spo2_drop"] = df["SpO2"].diff()
df["hr_change"] = df["HR"].diff()

df["spo2_rolling"] = df["SpO2"].rolling(10).mean()
df["hr_rolling"] = df["HR"].rolling(10).mean()

df = df.dropna()

df.head()

,Time [s],HR,PULSE,RESP,SpO2,spo2_drop,hr_change,spo2_rolling,hr_rolling
9,9,94,94.0,26,97.0,0.0,0.0,97.0,93.6
10,10,94,94.0,26,97.0,0.0,0.0,97.0,93.6
11,11,94,94.0,26,97.0,0.0,0.0,97.0,93.6
12,12,94,94.0,26,97.0,0.0,0.0,97.0,93.6
13,13,94,94.0,26,97.0,0.0,0.0,97.0,93.8


In [9]:
df["risk"] = (df["SpO2"] < 97).astype(int)

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

features = ["HR", "RESP", "spo2_rolling", "hr_rolling", "spo2_drop", "hr_change"]

X = df[features]
y = df["risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.8888888888888888


In [11]:
df["risk"].value_counts()

risk
0    399
1     51
Name: count, dtype: int64

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[70  4]
 [ 6 10]]
              precision    recall  f1-score   support

           0       0.92      0.95      0.93        74
           1       0.71      0.62      0.67        16

    accuracy                           0.89        90
   macro avg       0.82      0.79      0.80        90
weighted avg       0.88      0.89      0.89        90



In [13]:
model = RandomForestClassifier(class_weight="balanced")

In [14]:
model = RandomForestClassifier(class_weight="balanced")
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[71  3]
 [ 7  9]]
              precision    recall  f1-score   support

           0       0.91      0.96      0.93        74
           1       0.75      0.56      0.64        16

    accuracy                           0.89        90
   macro avg       0.83      0.76      0.79        90
weighted avg       0.88      0.89      0.88        90



In [16]:
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [17]:
y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[70  4]
 [ 6 10]]
              precision    recall  f1-score   support

           0       0.92      0.95      0.93        74
           1       0.71      0.62      0.67        16

    accuracy                           0.89        90
   macro avg       0.82      0.79      0.80        90
weighted avg       0.88      0.89      0.89        90



In [18]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=features)
print(importance.sort_values(ascending=False))

spo2_rolling    0.520662
hr_rolling      0.166469
RESP            0.163923
HR              0.080197
spo2_drop       0.052607
hr_change       0.016141
dtype: float64


In [19]:
model.predict_proba(X_test)

array([[0.68666667, 0.31333333],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.22333333, 0.77666667],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.81596296, 0.18403704],
       [1.        , 0.        ],
       [0.91666667, 0.08333333],
       [0.95333333, 0.04666667],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.01      , 0.99      ],
       [0.99666667, 0.00333333],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.98333333, 0.01666667],
       [0.12083333, 0.87916667],
       [0.99666667, 0.00333333],
       [0.35272222, 0.64727778],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.

In [20]:
proba = model.predict_proba(X_test)

proba[:10]

array([[0.68666667, 0.31333333],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.22333333, 0.77666667],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.81596296, 0.18403704],
       [1.        , 0.        ],
       [0.91666667, 0.08333333]])

In [21]:
risk_proba = proba[:, 1]

risk_proba[:10]

array([0.31333333, 0.        , 0.        , 0.        , 0.77666667,
       0.        , 0.        , 0.18403704, 0.        , 0.08333333])

In [22]:
y_pred_custom = (risk_proba > 0.3).astype(int)

In [23]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[70  4]
 [ 3 13]]
              precision    recall  f1-score   support

           0       0.96      0.95      0.95        74
           1       0.76      0.81      0.79        16

    accuracy                           0.92        90
   macro avg       0.86      0.88      0.87        90
weighted avg       0.92      0.92      0.92        90



In [24]:
y_pred_custom = (risk_proba > 0.1).astype(int)

In [25]:
print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[68  6]
 [ 1 15]]
              precision    recall  f1-score   support

           0       0.99      0.92      0.95        74
           1       0.71      0.94      0.81        16

    accuracy                           0.92        90
   macro avg       0.85      0.93      0.88        90
weighted avg       0.94      0.92      0.93        90



In [29]:
import joblib

joblib.dump(model, "hypoxia_model.pkl")
joblib.dump(features, "features.pkl")

['features.pkl']

In [30]:
import os
os.listdir()

['.ai-navigator',
 '.anaconda',
 '.bash_history',
 '.cache',
 '.conda',
 '.condarc',
 '.continuum',
 '.git',
 '.gitconfig',
 '.idlerc',
 '.ipynb_checkpoints',
 '.ipython',
 '.jupyter',
 '.mamba',
 '.matplotlib',
 '.ms-ad',
 '.packettracer',
 '.redhat',
 '.streamlit',
 '.UGENE_files',
 '.viminfo',
 '.VirtualBox',
 '.virtual_documents',
 '.vscode',
 '00_convert_rds',
 '31_xgboost_feature_importance_plot.R',
 '33_project_abstract.txt',
 '34_download_cnv.R',
 '35_prepare_cnv_matrix.R',
 '36_merge_multiomics_cnv.R',
 '37_build_multiomics_cnv_model.R',
 '38_cnv_kaplan_meier.R',
 '39_compare_cnv_vs_no_cnv.R',
 '40_train_test_validation__OLD.R',
 '41_train_test_kaplan_meier__OLD.R',
 '42_train_test_report.R',
 '43_repeated_train_test_cindex.R',
 '44_time_dependent_ROC.R',
 '45_final_model_summary.R',
 'anaconda3',
 'anaconda4',
 'anaconda4a',
 'anaconda5',
 'anaconda5a',
 'anaconda_projects',
 'AppData',
 'Application Data',
 'bidmc_01_Numerics.csv',
 'bidmc_01_Signals.csv',
 'bidmc_02_Breaths

In [31]:
import joblib

joblib.dump(model, "hypoxia_model.pkl")
joblib.dump(features, "features.pkl")

['features.pkl']